**Amazon DataSet**

In [1]:
import pandas as pd
import os

In [ ]:


def clean_data(df):
    # Drop duplicate rows in columns: 'text', 'label', 'hotel'
    df = df.drop_duplicates(subset=['text', 'label', 'hotel'])
    # Filter rows based on column: 'source'
    df = df[df['source'] == "amazon"]
    # Update 'label' values and add 'is_synthetic' column
    df['label'] = df['label'].replace({'OR': 'real', 'CG': 'fake'})
    df['is_synthetic'] = df['label'].apply(lambda x: 1 if x == 'fake' else 0)
    # Rename column 'label' to 'Binary_label'
    df = df.rename(columns={'label': 'Binary_label'})
    # Rename column 'hotel' to 'Category'
    df = df.rename(columns={'hotel': 'Category'})
    # Remove '_5' from values in 'Category' column
    df['Category'] = df['Category'].str.replace(r'_5$', '', regex=True)
    return df

# Loaded variable 'df' from URI: /home/xegi09mi/fake_reviews_prediction/Old_dataset/final_deceptive_opinion.csv
df = pd.read_csv(r'/home/xegi09mi/fake_reviews_prediction/Old_dataset/final_deceptive_opinion.csv')

df_clean = clean_data(df.copy())


In [4]:
df_clean.to_csv(r'/home/xegi09mi/fake_reviews_prediction/Old_dataset/Amazon_Human_VS_ComputerFake.csv', index=False)

**Hotel DataSet**

In [3]:

def clean_data(df):
    # Drop duplicate rows in columns: 'text', 'label', 'hotel'
    df = df.drop_duplicates(subset=['text', 'label', 'hotel'])
    # Filter rows based on column: 'source'
    df = df[df['source'] != "amazon"]
    # Rename column 'source' to 'domain '
    df = df.rename(columns={'source': 'domain '})
    # Fix KeyError by correcting column name
    df.insert(3, 'source', df['domain '])
    df["domain"] = "Hotel"
    # Drop column: 'domain '
    df = df.drop(columns=['domain '])
    # Rename column 'hotel' to 'Category'
    df = df.rename(columns={'hotel': 'Category'})
    # Rename column 'label' to 'Binary_label'
    df = df.rename(columns={'label': 'Binary_label'})
    df["is_synthetic"] = 0
    cols = ["Binary_label", "Category", "domain","text", "is_synthetic", "source"]  # desired order
    df = df[cols]
    return df

# Loaded variable 'df' from URI: /home/xegi09mi/fake_reviews_prediction/Old_dataset/final_deceptive_opinion.csv
df = pd.read_csv(r'/home/xegi09mi/fake_reviews_prediction/Old_dataset/final_deceptive_opinion.csv')

df_clean = clean_data(df.copy())


In [4]:
df_clean.to_csv(r'/home/xegi09mi/fake_reviews_prediction/Old_dataset/Hotel_Human_VS_HumanFake.csv', index=False)

**Removing 4 Fake to make it Balanced**

In [7]:


hotel_path = "/home/xegi09mi/fake_reviews_prediction/Old_dataset/Hotel_Human_VS_HumanFake.csv"

df_hotel = pd.read_csv(hotel_path)

print("Before:", df_hotel["Binary_label"].value_counts())

# Select 4 deceptive (fake) rows to drop
fake_mask = df_hotel["Binary_label"] == "deceptive"
fake_indices_to_drop = df_hotel[fake_mask].sample(n=4, random_state=42).index

df_balanced = df_hotel.drop(index=fake_indices_to_drop).reset_index(drop=True)

print("After:", df_balanced["Binary_label"].value_counts())

out_path = "/home/xegi09mi/fake_reviews_prediction/Old_dataset/Hotel_Human_VS_HumanFake.csv"
df_balanced.to_csv(out_path, index=False)

print("Balanced file saved to:", out_path)

Before: Binary_label
deceptive    800
truthful     796
Name: count, dtype: int64
After: Binary_label
truthful     796
deceptive    796
Name: count, dtype: int64
Balanced file saved to: /home/xegi09mi/fake_reviews_prediction/Old_dataset/Hotel_Human_VS_HumanFake.csv


**Hotel Real vs CG**

In [3]:

hotel_balanced_path = "/home/xegi09mi/fake_reviews_prediction/Old_dataset/Hotel_Human_VS_HumanFake.csv"
cg_path = "/home/xegi09mi/fake_reviews_prediction/Old_dataset/Hotel_CG_Reviews_final.csv"

df_hotel = pd.read_csv(hotel_balanced_path)
df_cg = pd.read_csv(cg_path)

# human real (truthful)
real_hotel = df_hotel[df_hotel["Binary_label"] == "truthful"].copy()

# sample same number of real and CG
n = min(len(real_hotel), len(df_cg))
real_sample = real_hotel.sample(n=n, random_state=42).copy()
cg_sample = df_cg.sample(n=n, random_state=42).copy()

# new binary label: real vs fake
real_sample["Binaery_label"] = "real"
cg_sample["Binaery_label"] = "fake"

df_humanReal_vs_CG = pd.concat([real_sample, cg_sample], ignore_index=True)

# keep consistent columns
cols = ["Binaery_label", "Category", "domain", "text", "is_synthetic", "source"]
df_humanReal_vs_CG = df_humanReal_vs_CG[cols]

out_hr_cg = "/home/xegi09mi/fake_reviews_prediction/Old_dataset/Hotel_HumanReal_VS_CG.csv"
df_humanReal_vs_CG.to_csv(out_hr_cg, index=False)
print("HumanRealVsCG saved to:", out_hr_cg)
print(df_humanReal_vs_CG["Binaery_label"].value_counts())

HumanRealVsCG saved to: /home/xegi09mi/fake_reviews_prediction/Old_dataset/Hotel_HumanReal_VS_CG.csv
Binaery_label
real    796
fake    796
Name: count, dtype: int64


**Hotel Human Real vs Mix**

*_398 CG and 398  Human Fake_*

In [4]:
# ...existing code...

# human fake (deceptive) from balanced hotel file
human_fake_hotel = df_hotel[df_hotel["Binary_label"] == "deceptive"].copy()

# choose 398 human fake and 398 CG
n_mix_each = 398
human_fake_sample = human_fake_hotel.sample(n=n_mix_each, random_state=42).copy()
cg_mix_sample = df_cg.sample(n=n_mix_each, random_state=43).copy()

# corresponding number of real = 398 + 398 = 796
real_for_mix = real_hotel.sample(n=2 * n_mix_each, random_state=44).copy()

# labels
real_for_mix["label_binary"] = "real"
human_fake_sample["label_binary"] = "fake"
cg_mix_sample["label_binary"] = "fake"

df_mix_fake = pd.concat(
    [real_for_mix, human_fake_sample, cg_mix_sample],
    ignore_index=True
)

cols = ["label_binary", "Category", "domain", "text", "is_synthetic", "source"]
df_mix_fake = df_mix_fake[cols]

out_hr_mix = "/home/xegi09mi/fake_reviews_prediction/Old_dataset/Hotel_HumanReal_VS_MixFake.csv"
df_mix_fake.to_csv(out_hr_mix, index=False)
print("HumanRealVsMix saved to:", out_hr_mix)
print(df_mix_fake["label_binary"].value_counts())

HumanRealVsMix saved to: /home/xegi09mi/fake_reviews_prediction/Old_dataset/Hotel_HumanReal_VS_MixFake.csv
label_binary
real    796
fake    796
Name: count, dtype: int64


**Hotel Human Real vs Human Fake label rename**

In [2]:
def clean_data(df):
    # Replace 'truthful' with 'real' and 'deceptive' with 'fake' in Binary_label
    df['Binary_label'] = df['Binary_label'].replace({'truthful': 'real', 'deceptive': 'fake'})
    return df

# Loaded variable 'df' from URI: /home/xegi09mi/fake_reviews_prediction/Datasets/Hotel_Human_VS_HumanFake.csv
df = pd.read_csv(r'/home/xegi09mi/fake_reviews_prediction/Datasets/Hotel_Human_VS_HumanFake.csv')



In [3]:
df_clean = clean_data(df.copy())
df_clean.to_csv(r'/home/xegi09mi/fake_reviews_prediction/Datasets/Hotel_Human_VS_HumanFake_relabelled.csv', index=False)